# CRM Reactivation Methodology
## Goal

The objective is to identify the customers with the highest probability of being reactivated after a long inactivity period.

In business terms, the company wants to answer this question:

> Among the customers who have already stopped buying for at least 2 years, which ones are the best targets for a reactivation campaign?

In [1]:
import numpy as np
import pandas as pd

In [2]:
client_df = pd.read_csv("TOOL_CLIENT.csv")
sales_df = pd.read_csv("TOOL_SALES.csv")

C:\Users\39346\AppData\Local\Temp\ipykernel_9772\4036812856.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv("TOOL_SALES.csv")


#  Dealing with datatypes

In [3]:
print(client_df.dtypes)
print(sales_df.dtypes)

CLIENT_ID               int64
CLIENT_CREATE DATE     object
REGION                 object
TRADE SECTOR            int64
N_EMPLOYEES             int64
ECONOMIC_POT          float64
ECO_POT_CLASS          object
RISK_CAT               object
dtype: object
YYYYMM             int64
ITEM_ID            int64
FLG_TOOL           int64
SALES_CHANNEL     object
NET              float64
UNIT              object
FAMILY_CODE       object
GROUP_CODE        object
CLIENT_ID          int64
CANCELLED         object
dtype: object


# Data cleaning

in CLIENT_TOOL:
CLIENT_ID               int64  ->  ok

CLIENT_CREATE DATE     object  ->  datetime64[ns]

REGION                 object  ->  string

TRADE SECTOR            int64  ->  ok

N_EMPLOYEES             int64  ->  ok

ECONOMIC_POT          float64  ->  ok

ECO_POT_CLASS          object  ->  string
 
RISK_CAT               object  -> string


In [9]:
client_df["CLIENT_CREATE DATE"] = pd.to_datetime(client_df["CLIENT_CREATE DATE"])
client_df["REGION"] = client_df["REGION"].astype("string")
#cambia raggruppando le regioni in macroaree
client_df["REGION"] = client_df["REGION"].fillna("unknown")
client_df["REGION"] = client_df["REGION"].astype("string")
client_df["ECO_POT_CLASS"] = client_df["ECO_POT_CLASS"].astype("string")
client_df["RISK_CAT"] = client_df["RISK_CAT"].astype("string")
# display(client_df.dtypes)
# display(client_df)


In Sales:
YYYYMM             int64   ->  datetime64[ns]

ITEM_ID            int64   ->  ok

FLG_TOOL           int64   ->  ok

SALES_CHANNEL     object   ->  string

NET              float64   ->  ok

UNIT              object   ->  string

FAMILY_CODE       object   ->  string

GROUP_CODE        object   ->  string

CLIENT_ID          int64   ->  ok

CANCELLED         object   ->  string

In [4]:

sales_df["UNIT"] = sales_df["UNIT"].astype("string")
sales_df["FAMILY_CODE"] = sales_df["FAMILY_CODE"].astype("string")

sales_df["YYYYMM"] = pd.to_datetime(sales_df["YYYYMM"], format="%Y%m")
sales_df["SALES_CHANNEL"] = sales_df["SALES_CHANNEL"].astype("string")
count = (sales_df["NET"] < 0).sum() #these rows will be removed as they represent returns, which are not relevant for our analysis
sales_df = sales_df[sales_df["NET"] >= 0]
sales_df = sales_df[sales_df["CANCELLED"].isna()]

## Inspecting transactions

standalone transactions are not usefull.
Transactions with negative values (not standalone) could provide usefull informations about a client.

In [5]:
#kp it

df_unique = sales_df[~sales_df["CLIENT_ID"].duplicated(keep=False)]
# df_unique = client_df[~client_df["column_name"].duplicated(keep=False)]
print(f"\"standalone\" transactions: {len(df_unique)}, over a total of: {len(sales_df)} that is {len(df_unique)/len(sales_df)*100:.2f}%")
print(f"Transactions with negative values: {len(sales_df[sales_df['NET'] < 0])}")

"standalone" transactions: 11712, over a total of: 1945557 that is 0.60%
Transactions with negative values: 0


Dates

In [6]:
cutoff_date = pd.Timestamp("2018-12-31")

min_registration_date = pd.Timestamp("2016-12-31")

feature_start = pd.Timestamp("2017-01-01")
feature_end = pd.Timestamp("2018-12-31")

outcome_start = pd.Timestamp("2019-01-01")
outcome_end = pd.Timestamp("2020-12-31")

# Inspecting clients

 - Since we have to take into account clients with at least 2 years of inactivity at 2018-12-31, in order for a client to be valid, it should be a client since at least 2016-12-31.

 - There are "Unuseful" clients who made only 1 transaction.
 - There are clients who stayed active after the cut-off date.
 

In [10]:


total_clients = len(client_df)
print(f"Total clients: {total_clients}")

clients_old_enough = client_df[client_df["CLIENT_CREATE DATE"] <= min_registration_date]

clients_not_old_enough = client_df[client_df["CLIENT_CREATE DATE"] > min_registration_date]

print(f"Over a total of {len(client_df)} clients, {len(clients_old_enough)} are old enough to be 24 months inactive by end-2018, {len(clients_not_old_enough)} are not.")
print(f" So, {len(clients_old_enough)/len(client_df)*100:.2f}% are \"valid\".")





Total clients: 93257
Over a total of 93257 clients, 64690 are old enough to be 24 months inactive by end-2018, 28567 are not.
 So, 69.37% are "valid".


Removing clients

In [11]:
# 1) eligible_registry_clients are clients old enough:
eligible_registry_clients = set(client_df.loc[client_df["CLIENT_CREATE DATE"] <= min_registration_date,"CLIENT_ID"].unique())

#clients who mad only one purchase
df_unique = sales_df[~sales_df["CLIENT_ID"].duplicated(keep=False)]
clients_valid_sales = set(df_unique["CLIENT_ID"].unique())
# print(len(clients_valid_sales))

#Diffence between the 2 sets
clients_valid_sales = eligible_registry_clients - clients_valid_sales
print("Clients old enough with at least one valid purchase:", len(clients_valid_sales))

#Still, we have to remove clients who were active in 2017-2018

# These clients are not part of the inactive pool, 
clients_active_2017_2018 = set(
    sales_df.loc[ sales_df["YYYYMM"].between(feature_start, feature_end), "CLIENT_ID" ].unique()
)

print("Clients active in 2017-2018:", len(clients_active_2017_2018))

final_set = clients_valid_sales - clients_active_2017_2018
print("Clients old enough, with at least one valid purchase, and inactive in 2017-2018:", len(final_set))
print(f"{len(final_set)} out of {total_clients}, which is {len(final_set)/total_clients*100:.2f}%")

# 5) activ  clients in 2019-2020
clients_activ_2019_2020 = set(
    sales_df.loc[
        sales_df["YYYYMM"].between(outcome_start, outcome_end),
        "CLIENT_ID"
    ].unique()
)

#we have to subtract clients which are not in the final set, as they are not part of the inactive pool or too young.
clients_reactivated_2019_2020 = final_set.intersection(clients_activ_2019_2020)

print(f"Clients reactivated in 2019-2020: {len(clients_reactivated_2019_2020)} which is {len(clients_reactivated_2019_2020)/total_clients*100:.2f}% of the total clients, and {len(clients_reactivated_2019_2020)/len(final_set)*100:.2f}% of the inactive clients.")


Clients old enough with at least one valid purchase: 58438
Clients active in 2017-2018: 59475
Clients old enough, with at least one valid purchase, and inactive in 2017-2018: 14371
14371 out of 93257, which is 15.41%
Clients reactivated in 2019-2020: 6384 which is 6.85% of the total clients, and 44.42% of the inactive clients.


Creating new dataframes to use from now on with only valid clients.

In [12]:
new_client_df = client_df[client_df["CLIENT_ID"].isin(final_set)]
new_sales_df = sales_df[sales_df["CLIENT_ID"].isin(final_set)]
model_df = new_client_df

creating the target and  getting the reactivation rate 

In [12]:
new_client_df["target"] = new_client_df["CLIENT_ID"].isin(clients_reactivated_2019_2020).astype(int)

print(f"{(new_client_df['target'] == 1).sum()/len(new_client_df)*100:.2f}% of the clients reactivated in 2019-2020")

44.42% of the clients reactivated in 2019-2020


C:\Users\39346\AppData\Local\Temp\ipykernel_10064\2295028371.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_client_df["target"] = new_client_df["CLIENT_ID"].isin(clients_reactivated_2019_2020).astype(int)


# Features engineering


Features from tool_client

In [13]:
feature_cols = [
    "REGION",
    "TRADE SECTOR",
    "N_EMPLOYEES",
    "ECONOMIC_POT",
    "ECO_POT_CLASS",
    "RISK_CAT"
]


Since how long a client is client? use age in months.

In [18]:
feature_cols = [
    "REGION",
    "TRADE SECTOR",
    "N_EMPLOYEES",
    "ECONOMIC_POT",
    "ECO_POT_CLASS",
    "RISK_CAT"
]

model_df["client_age_months"] = (
    (cutoff_date.year - model_df["CLIENT_CREATE DATE"].dt.year) * 12
    + (cutoff_date.month - model_df["CLIENT_CREATE DATE"].dt.month)
)

C:\Users\39346\AppData\Local\Temp\ipykernel_9772\755752423.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["client_age_months"] = (


Features that can be derived from transactions:

In [14]:
#Features that can be derived from transactions:
total_per_cust      = new_sales_df.groupby("CLIENT_ID")["NET"].sum()
tnsx_per_cust       = new_sales_df.groupby("CLIENT_ID")["NET"].count()
mean_spent_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].mean()
model_df["total_amount"]        = model_df["CLIENT_ID"].map(total_per_cust)
model_df["tsnx_p_cust"]         = model_df["CLIENT_ID"].map(tnsx_per_cust)
model_df["mean_spent_per_cust"] = model_df["CLIENT_ID"].map(mean_spent_per_cust)

C:\Users\39346\AppData\Local\Temp\ipykernel_9772\4140946482.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["total_amount"]        = model_df["CLIENT_ID"].map(total_per_cust)
C:\Users\39346\AppData\Local\Temp\ipykernel_9772\4140946482.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["tsnx_p_cust"]         = model_df["CLIENT_ID"].map(tnsx_per_cust)
C:\Users\39346\AppData\Local\Temp\ipykernel_9772\4140946482.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

Putting new features in the dataframe

In [15]:
model_df["total_amount"]        = model_df["CLIENT_ID"].map(total_per_cust)
model_df["tsnx_p_cust"]         = model_df["CLIENT_ID"].map(tnsx_per_cust)
model_df["mean_spent_per_cust"] = model_df["CLIENT_ID"].map(mean_spent_per_cust)

C:\Users\39346\AppData\Local\Temp\ipykernel_9772\262013609.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["total_amount"]        = model_df["CLIENT_ID"].map(total_per_cust)
C:\Users\39346\AppData\Local\Temp\ipykernel_9772\262013609.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["tsnx_p_cust"]         = model_df["CLIENT_ID"].map(tnsx_per_cust)
C:\Users\39346\AppData\Local\Temp\ipykernel_9772\262013609.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

In [35]:
def months_between(later, earlier):
    later = pd.to_datetime(later, errors="coerce")
    earlier = pd.to_datetime(earlier, errors="coerce")
    return (later.dt.year - earlier.dt.year) * 12 + (later.dt.month - earlier.dt.month)

def safe_mode(series):
    s = series.dropna()
    if len(s) == 0:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan

def shannon_entropy(series):
    s = series.dropna()
    if len(s) == 0:
        return np.nan
    p = s.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def compute_trend_slope(values):
    y = pd.Series(values).dropna().astype(float).values
    if len(y) < 2:
        return 0.0
    x = np.arange(len(y), dtype=float)
    return float(np.polyfit(x, y, 1)[0])


In [ ]:
# total_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].sum()
# avg_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].mean()
# std_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].std()
# max_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].max()

# model_df["TOTAL_NET_VALID_PRE"] = model_df["CLIENT_ID"].map(total_per_cust)
# model_df["AVG_ORDER_VALUE_VALID_PRE"] = model_df["CLIENT_ID"].map(avg_per_cust)
# model_df["STD_ORDER_VALUE_VALID_PRE"] = model_df["CLIENT_ID"].map(std_per_cust)
# model_df["MAX_ORDER_VALUE_VALID_PRE"] = model_df["CLIENT_ID"].map(max_per_cust)

# # last purchase before cutoff
# last_purchase_per_cust = new_sales_df.groupby("CLIENT_ID")["TXN_MONTH"].max()
# first_purchase_per_cust = new_sales_df.groupby("CLIENT_ID")["TXN_MONTH"].min()

# model_df["LAST_PURCHASE_PRE"] = model_df["CLIENT_ID"].map(last_purchase_per_cust)
# model_df["FIRST_PURCHASE_PRE"] = model_df["CLIENT_ID"].map(first_purchase_per_cust)

# # spend in last 6 / 12 months before last purchase
# tmp = new_sales_df[["CLIENT_ID", "TXN_MONTH", "NET"]].copy()
# tmp["LAST_PURCHASE_PRE"] = tmp["CLIENT_ID"].map(last_purchase_per_cust)

# tmp["MONTHS_BEFORE_LAST_PURCHASE"] = (
#     (tmp["LAST_PURCHASE_PRE"].dt.year - tmp["TXN_MONTH"].dt.year) * 12
#     + (tmp["LAST_PURCHASE_PRE"].dt.month - tmp["TXN_MONTH"].dt.month)
# )

# spend_6m = tmp.loc[tmp["MONTHS_BEFORE_LAST_PURCHASE"].between(0, 5)].groupby("CLIENT_ID")["NET"].sum()
# spend_12m = tmp.loc[tmp["MONTHS_BEFORE_LAST_PURCHASE"].between(0, 11)].groupby("CLIENT_ID")["NET"].sum()

# model_df["SPEND_LAST_6M_BEFORE_LAST_PURCHASE"] = model_df["CLIENT_ID"].map(spend_6m)
# model_df["SPEND_LAST_12M_BEFORE_LAST_PURCHASE"] = model_df["CLIENT_ID"].map(spend_12m)

# # spend trend slope on monthly net spend
# monthly_net = (
#     new_sales_df.groupby(["CLIENT_ID", "TXN_MONTH"])["NET"]
#     .sum()
#     .reset_index(name="NET_MONTH")
#     .sort_values(["CLIENT_ID", "TXN_MONTH"])
# )

# spend_trend = monthly_net.groupby("CLIENT_ID")["NET_MONTH"].apply(compute_trend_slope)
# model_df["SPEND_TREND_SLOPE_PRE"] = model_df["CLIENT_ID"].map(spend_trend)

# total_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].sum()
# avg_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].mean()
# std_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].std()
# max_per_cust = new_sales_df.groupby("CLIENT_ID")["NET"].max()

# model_df["TOTAL_NET_VALID_PRE"] = model_df["CLIENT_ID"].map(total_per_cust)
# model_df["AVG_ORDER_VALUE_VALID_PRE"] = model_df["CLIENT_ID"].map(avg_per_cust)
# model_df["STD_ORDER_VALUE_VALID_PRE"] = model_df["CLIENT_ID"].map(std_per_cust)
# model_df["MAX_ORDER_VALUE_VALID_PRE"] = model_df["CLIENT_ID"].map(max_per_cust)

# # last purchase before cutoff
# last_purchase_per_cust = new_sales_df.groupby("CLIENT_ID")["TXN_MONTH"].max()
# first_purchase_per_cust = new_sales_df.groupby("CLIENT_ID")["TXN_MONTH"].min()

# model_df["LAST_PURCHASE_PRE"] = model_df["CLIENT_ID"].map(last_purchase_per_cust)
# model_df["FIRST_PURCHASE_PRE"] = model_df["CLIENT_ID"].map(first_purchase_per_cust)

# # spend in last 6 / 12 months before last purchase
# tmp = new_sales_df[["CLIENT_ID", "TXN_MONTH", "NET"]].copy()
# tmp["LAST_PURCHASE_PRE"] = tmp["CLIENT_ID"].map(last_purchase_per_cust)

# tmp["MONTHS_BEFORE_LAST_PURCHASE"] = (
#     (tmp["LAST_PURCHASE_PRE"].dt.year - tmp["TXN_MONTH"].dt.year) * 12
#     + (tmp["LAST_PURCHASE_PRE"].dt.month - tmp["TXN_MONTH"].dt.month)
# )

# spend_6m = tmp.loc[tmp["MONTHS_BEFORE_LAST_PURCHASE"].between(0, 5)].groupby("CLIENT_ID")["NET"].sum()
# spend_12m = tmp.loc[tmp["MONTHS_BEFORE_LAST_PURCHASE"].between(0, 11)].groupby("CLIENT_ID")["NET"].sum()

# model_df["SPEND_LAST_6M_BEFORE_LAST_PURCHASE"] = model_df["CLIENT_ID"].map(spend_6m)
# model_df["SPEND_LAST_12M_BEFORE_LAST_PURCHASE"] = model_df["CLIENT_ID"].map(spend_12m)

# # spend trend slope on monthly net spend
# monthly_net = (
#     new_sales_df.groupby(["CLIENT_ID", "TXN_MONTH"])["NET"]
#     .sum()
#     .reset_index(name="NET_MONTH")
#     .sort_values(["CLIENT_ID", "TXN_MONTH"])
# )

# spend_trend = monthly_net.groupby("CLIENT_ID")["NET_MONTH"].apply(compute_trend_slope)
# model_df["SPEND_TREND_SLOPE_PRE"] = model_df["CLIENT_ID"].map(spend_trend)

C:\Users\39346\AppData\Local\Temp\ipykernel_10064\2718154620.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["TOTAL_NET_VALID_PRE"] = model_df["CLIENT_ID"].map(total_per_cust)
C:\Users\39346\AppData\Local\Temp\ipykernel_10064\2718154620.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["AVG_ORDER_VALUE_VALID_PRE"] = model_df["CLIENT_ID"].map(avg_per_cust)
C:\Users\39346\AppData\Local\Temp\ipykernel_10064\2718154620.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy 

KeyError: 'Column not found: TXN_MONTH'

In [33]:
print(model_df["total_amount"].isna().sum())
print(model_df["tsnx_p_cust"].isna().sum())
print(model_df["mean_spent_per_cust"].isna().sum())
# len(model_df)
df = model_df.sort_values(by="tsnx_p_cust", ascending=True)
df

6192
6192
6192


,CLIENT_ID,CLIENT_CREATE DATE,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT,target,client_age_months,total_amount,tsnx_p_cust,mean_spent_per_cust
54753,54760,2014-07-23,FG,11000,2,3019.02,E,3d,1,53,158.40,2.0,79.200
53676,53689,2014-04-03,AP,22000,2,4528.46,E,5d,1,56,169.86,2.0,84.930
53678,53683,2014-04-03,FI,14400,4,1815.72,E,5d,1,56,100.55,2.0,50.275
29664,29475,2007-05-27,SP,33800,35,7695.00,D,3d,1,139,1596.17,2.0,798.085
13596,7547,2005-11-15,VA,21100,4,11297.08,D,3d,1,157,122.21,2.0,61.105
...,...,...,...,...,...,...,...,...,...,...,...,...,...
64601,64600,2016-12-19,TV,21100,4,11297.08,D,T8,0,24,NaN,NaN,NaN
64617,64635,2016-12-20,MI,11000,3,4528.53,E,T8,0,24,NaN,NaN,NaN
64630,64623,2016-12-20,TO,42200,13,3378.00,E,3d,0,24,NaN,NaN,NaN
64668,64662,2016-12-22,PD,11000,8,10884.35,D,3d,0,24,NaN,NaN,NaN


In [18]:




client_df

    #    "N_EMPLOYEES",
    #     "ECONOMIC_POT",
    #     "client_age_months",
    #     "purchase_tenure_months",
    #     "recency_months",
    #     "dormancy_over_24m",
    #     "months_from_create_to_first_purchase",
    #     "purchase_density",
    #     "raw_txn_count",
    #     "cancelled_txn_count",
    #     "nonpositive_txn_count",
    #     "cancelled_share",
    #     "nonpositive_share",
    #     "pre12_purchase_count",
    #     "pre12_active_months",
    #     "pre12_total_net",
    #     "pre12_avg_net",
    #     "pre12_median_net",
    #     "pre12_max_net",
    #     "pre12_std_net",
    #     "pre12_unique_items",
        # "pre12_unique_families",
        # "pre12_unique_groups",
        # "pre12_unique_channels",
        # "pre12_tool_txn_share",
        # "pre12_avg_monthly_net",
        # "pre12_txn_per_active_month",
        # "last_month_total_net",
        # "last_month_txn_count",
        # "last_month_avg_net",
        # "last_month_max_net",
        # "last_month_unique_items",
        # "last_month_unique_families",
        # "last_month_unique_groups",
        # "last_month_unique_channels",
        # "last_month_tool_txn_share",
        # "last_month_share_of_pre12_net",
        # "last_month_share_of_pre12_txn",
        # "pre12_channel_share_A",
        # "pre12_channel_share_B",
        # "pre12_channel_share_C",
        # "pre12_channel_share_D",

,CLIENT_ID,CLIENT_CREATE DATE,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT
0,9306,2005-11-15,BZ,11000,6,8659.81,D,3d
1,939,2005-11-15,LE,15500,2,681.26,E,3d
2,8321,2005-11-15,LE,15500,2,681.26,E,T8
3,4174,2005-11-15,BZ,15400,1,494.45,E,3d
4,12765,2005-11-15,BZ,15400,1,494.45,E,3b
...,...,...,...,...,...,...,...,...
93252,93242,2021-12-23,PD,14000,1,1933.89,E,5d
93253,93237,2021-12-23,TO,11000,5,7547.54,D,5d
93254,93257,2021-12-23,RE,11000,1,1509.51,E,5d
93255,93224,2021-12-23,RM,13500,4,8864.84,D,5d


In [20]:
# df = client_df.sort_values(by="client_age_months", ascending=False)
# df

In [34]:

dcjo =  new_sales_df[new_sales_df["CLIENT_ID"] == 29475]
dcjo.sort_values(by="YYYYMM", ascending=False)



,YYYYMM,ITEM_ID,FLG_TOOL,SALES_CHANNEL,NET,UNIT,FAMILY_CODE,GROUP_CODE,CLIENT_ID,CANCELLED
1621664,2020-11-01,2704,0,A,638.47,P,XBXE2GE,XBXE2GE0104,29475,NaN
1583893,2020-10-01,2704,0,A,957.70,P,XBXE2GE,XBXE2GE0104,29475,NaN


In [23]:
# df = client_df.sort_values(by="tsnx_p_cust", ascending=False)
# df

In [25]:
# client_df[client_df["target"] == 1]

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

In [25]:
X

,CLIENT_ID,CLIENT_CREATE DATE,REGION,TRADE SECTOR,N_EMPLOYEES,ECONOMIC_POT,ECO_POT_CLASS,RISK_CAT,client_age_months,total_amount,tsnx_p_cust,mean_spent_per_cust
1,939,2005-11-15,LE,15500,2,681.26,E,3d,157,NaN,NaN,NaN
5,10364,2005-11-15,BO,22000,4,9056.92,D,3d,157,617.37,6.0,102.895000
6,11767,2005-11-15,BL,22100,1,2264.23,E,3d,157,NaN,NaN,NaN
7,9868,2005-11-15,MI,36400,12,8217.00,D,5a,157,221.68,4.0,55.420000
13,858,2005-11-15,CN,14400,3,1361.79,E,3d,157,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
64668,64662,2016-12-22,PD,11000,8,10884.35,D,3d,24,NaN,NaN,NaN
64671,64679,2016-12-23,AL,22100,1,2264.23,E,3d,24,154.82,2.0,77.410000
64678,64681,2016-12-23,SI,14400,1,453.93,E,T8,24,NaN,NaN,NaN
64682,64672,2016-12-23,RM,11000,1,1509.51,E,3d,24,79.24,2.0,39.620000


In [26]:

from sklearn.model_selection import train_test_split

feature_cols = [
    "REGION",
    "TRADE SECTOR",
    "N_EMPLOYEES",
    "ECONOMIC_POT",
    "ECO_POT_CLASS",
    "RISK_CAT",
    "client_age_months"
]

model_df["target"] = model_df["CLIENT_ID"].isin(clients_reactivated_2019_2020).astype(int)


X = model_df.copy()
y = X.drop(columns=["target"])
y = model_df["target"].copy()

numeric_features = [
    "TRADE SECTOR",
    "N_EMPLOYEES",
    "ECONOMIC_POT",
    "client_age_months"
]

categorical_features = [
    "REGION",
    "ECO_POT_CLASS",
    "RISK_CAT"
]


numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Train target mean:", y_train.mean())
print("Test target mean:", y_test.mean())

clf.fit(X_train, y_train)

C:\Users\39346\AppData\Local\Temp\ipykernel_9772\219974657.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  model_df["target"] = model_df["CLIENT_ID"].isin(clients_reactivated_2019_2020).astype(int)


X_train: (11496, 13)
X_test: (2875, 13)
Train target mean: 0.44424147529575503
Test target mean: 0.44417391304347825


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['TRADE SECTOR',
                                                   'N_EMPLOYEES',
                                                   'ECONOMIC_POT',
                                                   'client_age_months']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['REGION', 'ECO_POT_CLASS',
                                                   'RISK_CAT'])])),
                ('model', LogisticRegression(max_iter=2000, random_state=42))])

In [27]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

AUC: 0.7293330151334432

Classification report:
              precision    recall  f1-score   support

           0       0.74      0.62      0.67      1598
           1       0.60      0.73      0.66      1277

    accuracy                           0.67      2875
   macro avg       0.67      0.67      0.67      2875
weighted avg       0.68      0.67      0.67      2875

Confusion matrix:
[[984 614]
 [348 929]]


In [28]:
import pandas as pd
import numpy as np

# Get transformed feature names from the fitted preprocessor
feature_names = clf.named_steps["preprocessor"].get_feature_names_out()

# Get logistic regression coefficients
coefs = clf.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
    "odds_ratio": np.exp(coefs)
}).sort_values("abs_coefficient", ascending=False)

display(coef_df.head(20))

,feature,coefficient,abs_coefficient,odds_ratio
134,cat__RISK_CAT_T8,-1.983333,1.983333,0.137610
129,cat__RISK_CAT_4d,-1.322719,1.322719,0.266410
118,cat__RISK_CAT_2a,0.962137,0.962137,2.617283
124,cat__RISK_CAT_3c,0.946046,0.946046,2.575505
28,cat__REGION_CL,-0.937430,0.937430,0.391633
32,cat__REGION_CS,0.819254,0.819254,2.268806
133,cat__RISK_CAT_5d,0.726463,0.726463,2.067755
63,cat__REGION_NU,0.649531,0.649531,1.914643
123,cat__RISK_CAT_3b,0.635353,0.635353,1.887688
23,cat__REGION_CA,0.632519,0.632519,1.882346
